# LEVELS Benchmark — Level 2: Workflow

Multi-step pipelines combining ≥2 data-management categories (e.g. search → download →
summarize, filter → report, ingest → validate). Run **non-isolated** (see `run_level`), so
tasks sharing a list may see each other's conversational context; tasks that need guaranteed
clean state use `setup_query` or an explicit `steps` list of turns run against one shared
explorer (`task.steps`, see `run_task`). Scored by `score_level2()`: every tool in
`expected_tools` must be used at least once (not any-of, like Level 1), plus an LLM-judged
process-quality score against `success_criteria`. Ground-truth counts/fields below were
hand-verified against `data/oceans_11/*.db` directly with sqlite3, the same way Level 1's
were verified via the DSI CLI.

#### How `score_level2()` scores a task

| Component | What it checks | Pass condition | Contributes to |
|---|---|---|---|
| **Tool coverage** (`tools_ok`) | Did the agent call *every* tool listed in `expected_tools`? | ALL required tools must appear in `tool_calls` (unlike Level 1, where only *one* correct tool is needed) | Correctness |
| **Quantitative check** (`quant_ok`) | Does the expected number (`expected_count`) show up in the tool results or final response? | Regex match for the number as a `count:`/`missing_count:` field or a standalone number in the combined text blob | Correctness |
| **Keyword check** (`keyword_ok`) | Do all required facts (`response_keywords`) appear in the response/tool output? | ALL keywords must be present (case-insensitive) — every one, not just any one | Correctness |
| **Forbidden-keyword check** (`forbidden_ok`) | Did the agent avoid stating any wrong/distractor facts (`forbidden_keywords`)? | None of the forbidden terms may appear | Correctness |
| **Non-empty check** (`nonempty`) | Did the agent actually produce a final answer? | `response` must be non-blank | Correctness |
| **Correctness** (`result.passed`) | All five checks above | `tools_ok AND quant_ok AND keyword_ok AND forbidden_ok AND nonempty` | Binary 0.0 / 1.0 — **50% of final score** |
| **Process quality** (`judge_scores["process_quality_score"]`) | Was the *method* good — efficient, well-ordered, matching the `success_criteria` rubric — regardless of whether the final answer was right? | LLM judge (both models, scores averaged) rates 0.0 (ignored rubric / haphazard) to 1.0 (clean, sound multi-step approach) | Continuous 0.0–1.0 — **50% of final score** |

**Final score formula:**
- If the task has `success_criteria` set: `score = 0.5 × correctness (0 or 1) + 0.5 × process_quality_score`
- If no `success_criteria`: `score = correctness` only (0.0 or 1.0)

**Important nuance:** `result.passed` reflects **correctness only** — a task can "pass" (did all the right steps, got the right facts) while still scoring well below 1.0 overall if the LLM judge dings it for a messy/inefficient process. That's deliberate: it lets the benchmark distinguish "got it right, cleanly" from "got it right, but wastefully."

**Session isolation:** every task runs in a fresh agent session. Tasks needing multi-turn context already define their own `steps`, so isolation is what allows them to run in parallel. To reproduce the original single shared session per model, pass `share_session=True` to `run_benchmark`. Only models then run in parallel.

Shared harness code (scoring, parallel runner, reports) lives in `levels_bench/`, and the task
definitions in `levels_bench/tasks/`. Each (model, task) pair runs as an independent job in a
pool of worker processes, so `max_workers` runs proceed at once. Every finished result is
appended to `levels_benchmark_results/<run_id>_<label>.partial.jsonl` as it completes. If a run
is interrupted, call `run_benchmark(..., resume=True)` to run only what's missing.

## Setup

In [1]:
# Run once per environment (not needed on every run):
# %pip install -q -r requirements.txt
# %pip install -q --no-build-isolation -e ../..

In [2]:
from levels_bench import init, run_benchmark, flatten, results_to_dataframe, load_results, reports, debug_task
from levels_bench.tasks import LEVEL2_TASKS, get_task

cfg = init(
    # models_to_run=["aws-gov.gpt-5.4"],   # default: every model in levels_bench.config.POSSIBLE_MODELS
    # judge_models=[...],                   # default: same list; judged scores are averaged across them
    max_workers=4,                          # concurrent agent runs; raise if the portal's rate limit allows
    task_timeout_sec=180,                   # per agent turn
    # skip_network=True,                    # skip tasks that download from oceans11.lanl.gov
)

Working directory: /Users/hhn/lanl/dsi/tools/ai_search
Run id: 20261022_063223 | models: ['aws-gov.gpt-5.4', 'anthropic.claude-sonnet-4-5-20250929-v1:0'] | judges: ['aws-gov.gpt-5.4', 'anthropic.claude-sonnet-4-5-20250929-v1:0']
Workers: 4 | turn timeout: 180s | skip_network: False


In [3]:
reports.task_table(LEVEL2_TASKS)

,task_id,level,category,turns,network,variants,query
0,L2_search_download_summarize,2,Analysis & Summarization,1,True,5,Find the NIF simulation dataset in the Ocean 1...
1,L2_physics_contact_report,2,Metadata Generation,1,False,5,List the titles of every physics-themed datase...
2,L2_poisson_field_completeness,2,Data Preparation,1,False,5,Load the Poisson database and check whether it...
3,L2_cross_db_record_compare,2,Analysis & Summarization,3,False,5,Compare the number of discrete records in the ...
4,L2_dsi_database_link_audit,2,Metadata Generation,1,False,5,"For every dataset in the Ocean 11 catalog, che..."
5,L2_poisson_args_filename_facts,2,Data Preparation,1,True,5,Download the Poisson dataset's simulation-argu...
6,L2_poisson_glossary_audit_1,2,Data Preparation,1,False,5,The Poisson dataset has a field glossary file ...
7,L2_poisson_glossary_audit_2,2,Data Preparation,1,False,5,The Poisson dataset has a field glossary (pois...
8,L2_simulation_tag_contacts,2,Metadata Generation,1,False,5,Find every dataset in the Ocean 11 catalog who...
9,L2_nif_dens_max_extremum,2,Analysis & Summarization,1,False,5,"Across all 20 timesteps of the NIF simulation,..."


## Run

In [4]:
l2_results = flatten(run_benchmark(LEVEL2_TASKS, "level2_workflow"))
# To analyze an earlier run without re-running it:
# l2_results = load_results("level2_workflow")
l2_df = results_to_dataframe(l2_results)
l2_df

level2_workflow: 12 tasks x 2 models = 24 runs (24 to run, 4 workers)
[ 1/24] FAIL    aws-gov.gpt-5.4 | L2_physics_contact_report  score=0.43  17s   (elapsed 0m18s, ~7m09s left)
[ 2/24] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_physics_contact_report  score=0.97  18s   (elapsed 0m20s, ~3m45s left)
[ 3/24] PASS    aws-gov.gpt-5.4 | L2_search_download_summarize  score=0.95  19s   (elapsed 0m21s, ~2m27s left)
[ 4/24] PASS    aws-gov.gpt-5.4 | L2_poisson_field_completeness  score=0.95  10s   (elapsed 0m28s, ~2m24s left)
[ 5/24] PASS    aws-gov.gpt-5.4 | L2_cross_db_record_compare  score=0.73  16s   (elapsed 0m37s, ~2m22s left)
[ 6/24] PASS    aws-gov.gpt-5.4 | L2_dsi_database_link_audit  score=1.00  8s   (elapsed 0m45s, ~2m15s left)
[ 7/24] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_search_download_summarize  score=0.83  45s   (elapsed 0m47s, ~1m55s left)
[ 8/24] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_poisson_field_completeness  score=0.96  29s   (e

,model,task_id,query,prompt_style,base_task_id,level,category,status,passed,score,...,expects_abstention,hallucination_detected,checkpoint_opportunity_count,high_stakes_action_count,hedge_language_count,failure_category,judge_process_quality_score,judge_process_quality_reasoning,judge_efficiency_score,judge_efficiency_reasoning
0,aws-gov.gpt-5.4,L2_search_download_summarize,Find the NIF simulation dataset in the Ocean 1...,standard,L2_search_download_summarize,2,Analysis & Summarization,PASS,True,0.955,...,False,False,1,1,0,None,0.910,[aws-gov.gpt-5.4] The workflow efficiently fou...,0.560,[aws-gov.gpt-5.4] The agent completed the task...
1,aws-gov.gpt-5.4,L2_physics_contact_report,List the titles of every physics-themed datase...,standard,L2_physics_contact_report,2,Metadata Generation,FAIL,False,0.435,...,False,False,1,0,1,MODEL_REASONING,0.870,[aws-gov.gpt-5.4] The workflow efficiently que...,0.460,[aws-gov.gpt-5.4] The agent used two SQL queri...
2,aws-gov.gpt-5.4,L2_poisson_field_completeness,Load the Poisson database and check whether it...,standard,L2_poisson_field_completeness,2,Data Preparation,PASS,True,0.953,...,False,False,0,0,0,None,0.905,[aws-gov.gpt-5.4] The agent appropriately load...,0.360,[aws-gov.gpt-5.4] The agent reached the correc...
3,aws-gov.gpt-5.4,L2_cross_db_record_compare,Compare the number of discrete records in the ...,standard,L2_cross_db_record_compare,2,Analysis & Summarization,PASS,True,0.730,...,False,False,0,0,0,None,0.460,[aws-gov.gpt-5.4] The agent reached the correc...,0.280,[aws-gov.gpt-5.4] The agent got the needed cou...
4,aws-gov.gpt-5.4,L2_dsi_database_link_audit,"For every dataset in the Ocean 11 catalog, che...",standard,L2_dsi_database_link_audit,2,Metadata Generation,PASS,True,1.000,...,False,False,0,0,0,None,1.000,[aws-gov.gpt-5.4] The agent directly queried t...,1.000,[aws-gov.gpt-5.4] The agent matched the refere...
5,aws-gov.gpt-5.4,L2_poisson_args_filename_facts,Download the Poisson dataset's simulation-argu...,standard,L2_poisson_args_filename_facts,2,Data Preparation,PASS,True,0.995,...,False,False,1,1,0,None,0.990,[aws-gov.gpt-5.4] The agent followed the inten...,0.425,[aws-gov.gpt-5.4] The agent completed the task...
6,aws-gov.gpt-5.4,L2_poisson_glossary_audit_1,The Poisson dataset has a field glossary file ...,standard,L2_poisson_glossary_audit_1,2,Data Preparation,PASS,True,0.905,...,False,False,0,0,0,None,0.810,[aws-gov.gpt-5.4] The workflow did read the gl...,0.510,[aws-gov.gpt-5.4] The agent did obtain the dat...
7,aws-gov.gpt-5.4,L2_poisson_glossary_audit_2,The Poisson dataset has a field glossary (pois...,standard,L2_poisson_glossary_audit_2,2,Data Preparation,FAIL,False,0.295,...,False,False,0,0,0,TOOL_FRAMEWORK,0.590,[aws-gov.gpt-5.4] The final answer matches the...,0.275,[aws-gov.gpt-5.4] The agent loaded the target ...
8,aws-gov.gpt-5.4,L2_simulation_tag_contacts,Find every dataset in the Ocean 11 catalog who...,standard,L2_simulation_tag_contacts,2,Metadata Generation,FAIL,False,0.497,...,False,False,0,0,0,UNKNOWN,0.995,[aws-gov.gpt-5.4] The agent used the correct s...,0.480,[aws-gov.gpt-5.4] The agent got the needed inf...
9,aws-gov.gpt-5.4,L2_nif_dens_max_extremum,"Across all 20 timesteps of the NIF simulation,...",standard,L2_nif_dens_max_extremum,2,Analysis & Summarization,PASS,True,0.992,...,False,False,0,0,0,None,0.985,[aws-gov.gpt-5.4] The agent correctly located ...,0.410,[aws-gov.gpt-5.4] The agent reached the correc...


## Results

In [5]:
reports.model_summary(l2_df, extra_judge_cols=['judge_process_quality_score'])

### Overall summary by model

,pass_rate,avg_score,avg_efficiency,avg_tokens,avg_elapsed_sec,n_tasks,n_errors,process_quality_score
model,,,,,,,,
anthropic.claude-sonnet-4-5-20250929-v1:0,0.92,0.91,0.44,5623.67,16.18,12,0,0.90
aws-gov.gpt-5.4,0.75,0.81,0.49,3048.17,5.82,12,0,0.88


### Pass rate by model and category

category,Analysis & Summarization,Data Discovery,Data Preparation,Metadata Generation
model,,,,
anthropic.claude-sonnet-4-5-20250929-v1:0,1.0,1.0,0.75,1.00
aws-gov.gpt-5.4,1.0,1.0,0.75,0.33


### Pass/fail matrix by task and model

model,anthropic.claude-sonnet-4-5-20250929-v1:0,aws-gov.gpt-5.4
task_id,,
L2_cross_db_record_compare,PASS,PASS
L2_dsi_database_link_audit,PASS,PASS
L2_most_recent_drilldown,PASS,PASS
L2_nif_dens_max_extremum,PASS,PASS
L2_nif_num_arrays_tie,PASS,PASS
L2_physics_contact_report,PASS,FAIL
L2_poisson_args_filename_facts,PASS,PASS
L2_poisson_field_completeness,PASS,PASS
L2_poisson_glossary_audit_1,PASS,PASS


In [6]:
reports.failures(l2_df)

### Non-passing runs (4)

,model,task_id,status,score,notes,failure_category
1,aws-gov.gpt-5.4,L2_physics_contact_report,FAIL,0.435,Expected count 5 not found in tool results/response,MODEL_REASONING
7,aws-gov.gpt-5.4,L2_poisson_glossary_audit_2,FAIL,0.295,"Missing required tools for this workflow: ['python_repl_tool'] (called: ['load_dsi_tool', 'query_dsi_tool'])",TOOL_FRAMEWORK
8,aws-gov.gpt-5.4,L2_simulation_tag_contacts,FAIL,0.497,Expected count 5 not found in tool results/response,UNKNOWN
19,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_poisson_glossary_audit_2,FAIL,0.448,"Missing required tools for this workflow: ['python_repl_tool'] (called: ['query_dsi_tool', 'load_dsi_tool'])",UNKNOWN


In [7]:
reports.timing_breakdown(l2_results)

### Where does the time go?

#### Top-level phases (real wall-clock, no double counting)

,total_sec,mean_sec,count
phase_group,,,
step,264.121,9.433,28
score,155.597,1.621,96


#### By model x top-level phase group (total seconds)

phase_group,score,step
model,,
anthropic.claude-sonnet-4-5-20250929-v1:0,78.71,194.18
aws-gov.gpt-5.4,76.89,69.94


#### Nested inside steps (already counted above)

,total_sec,mean_sec,count
phase,,,
llm_call,261.832,2.424,108
tool:download_file_tool,1.608,0.268,6
tool:query_dsi_tool,0.273,0.005,53
tool:load_dsi_tool,0.092,0.005,18
tool:python_repl_tool,0.020,0.003,7


## Inspect / debug

Transcripts are recorded for every run. `debug_task` reruns one task in this process with live output.

In [8]:
# reports.transcript(l2_results, "L2_physics_contact_report")
# tr = debug_task(get_task("L2_physics_contact_report"), model="anthropic.claude-sonnet-4-5-20250929-v1:0")